# Component 02: AST bird classification
J26-IT-359. This notebook has no training results. Use a fresh Colab runtime (GPU preferred; CPU smoke test supported).
Checkpoint: `MIT/ast-finetuned-audioset-10-10-0.4593`. A new 10-class classification head replaces the pretrained task head. Head initialization warnings are expected. Training and test evaluation are separately opt-in.


## 1. Dependencies and runtime
Install first. If Colab requests a runtime restart, restart and continue at imports. Keep Colab's matched GPU torch/torchaudio installation; do not install CPU torch wheels. All downloads occur only when you run this notebook.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q',
 'transformers==4.57.1','optuna==4.5.0','scikit-learn==1.7.2','librosa==0.11.0',
 'soundfile==0.13.1','pandas>=2.2,<3','matplotlib>=3.8,<4',
 'safetensors>=0.4.3','huggingface-hub>=0.34,<1'])


In [ ]:
import os
from contextlib import nullcontext
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
from pathlib import Path
from datetime import datetime,timezone
import json,math,random,hashlib,zipfile,shutil,gc,platform,importlib.metadata,io
import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix,ConfusionMatrixDisplay
import optuna
from transformers import AutoConfig,AutoFeatureExtractor,Wav2Vec2ForSequenceClassification,ASTForAudioClassification,get_cosine_schedule_with_warmup
from huggingface_hub import HfApi
from safetensors.torch import load_file
from google.colab import drive,files
assert tuple(int(v) for v in torch.__version__.split('+')[0].split('.')[:2]) >= (2,6), 'Use a fresh torch >=2.6 Colab runtime'
SEED=42
def seed_everything(seed):
 random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
 if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
 torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
 torch.use_deterministic_algorithms(True,warn_only=True)
seed_everything(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
AMP_DTYPE = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if AMP_ENABLED else torch.float32
def amp_context():
 return torch.autocast(device_type='cuda',dtype=AMP_DTYPE) if AMP_ENABLED else nullcontext()
print(torch.cuda.get_device_name(0) if AMP_ENABLED else 'CPU', 'mixed precision:', AMP_ENABLED)


## 2. Drive and run settings
Edit ZIP_ON_DRIVE. Each model uses a separate timestamped output directory. Practical mode tunes on one training-only grouped holdout; optional full CV uses three folds. Three trials do not exhaust the search space. OOM trials are pruned; batch sizes are never silently changed.

For a small CPU experiment set `CPU_SMOKE_TEST=True` here before running settings. Leave execution switches off until checks finish. This uses 20 clean training clips and 10 disjoint training-only validation clips, one trial, batch size 1 and at most one epoch. Original validation/test are not scored in smoke mode. Results are preliminary, not comparable to full-model results. CPU full training remains possible with smoke mode off but can be extremely slow.


In [ ]:
drive.mount('/content/drive')
MODEL_KIND='ast'
CHECKPOINT='MIT/ast-finetuned-audioset-10-10-0.4593'
ZIP_ON_DRIVE=Path('/content/drive/MyDrive/J26-IT-359/colab_training_package.zip')  # EDIT
DRIVE_ROOT=Path('/content/drive/MyDrive/J26-IT-359/model_runs')
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
CPU_SMOKE_TEST=False  # Opt in before running this settings cell; does not start training.
GRADIENT_CHECKPOINTING=False  # Optional GPU-only optimization; disabled on CPU.
RUN_DIR=(DRIVE_ROOT/MODEL_KIND/'cpu_smoke_test'/RUN_ID) if CPU_SMOKE_TEST else (DRIVE_ROOT/MODEL_KIND/RUN_ID)
RUN_DIR.mkdir(parents=True,exist_ok=False)
N_TRIALS=3
FULL_GROUPED_CV=False
BATCH_CHOICES=[4,8,16]  # If every trial OOMs, start a new run with [4]
ACCUMULATION_STEPS=4
EARLY_STOPPING_PATIENCE=2
MIN_F1_IMPROVEMENT=1e-4
NUM_WORKERS=0
RUN_TUNING=False  # Enable explicitly when ready for GPU computation
RUN_FINAL_TRAINING=False
VERIFY_AUDIO_HASHES=True
if CPU_SMOKE_TEST:
 N_TRIALS=1; FULL_GROUPED_CV=False; BATCH_CHOICES=[1]; ACCUMULATION_STEPS=1
 print('PRELIMINARY CPU SMOKE TEST: not comparable to full-model results.')
assert ZIP_ON_DRIVE.is_file(),'Upload ZIP to Drive and edit ZIP_ON_DRIVE'
def sha256(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for block in iter(lambda:f.read(1048576),b''):h.update(block)
 return h.hexdigest()
def save_json(path,value):Path(path).write_text(json.dumps(value,indent=2,default=str),encoding='utf-8')
ZIP_SHA256=sha256(ZIP_ON_DRIVE)
LOCAL=Path('/content')/('component02_'+MODEL_KIND+'_'+RUN_ID)
with zipfile.ZipFile(ZIP_ON_DRIVE) as z:
 assert len(z.namelist())==len(set(z.namelist()))
 assert shutil.disk_usage('/content').free>sum(i.file_size for i in z.infolist())+2*1024**3,'Insufficient disk'
 for item in z.infolist():
  assert (LOCAL/item.filename).resolve().is_relative_to(LOCAL.resolve()),'Unsafe ZIP path'
  assert ((item.external_attr >> 16)&0o170000)!=0o120000,'ZIP symlinks not allowed'
 assert z.testzip() is None,'ZIP CRC failure'
 z.extractall(LOCAL)
ROOT=LOCAL/'colab_training_package'
print(ROOT)


## 3. Verify package schema and authoritative split
Test metadata is inspected for integrity only; test waveforms and metrics are not used for tuning. No random re-split of train/val/test is allowed. All package quality flags are retained.


In [ ]:
frames={s:pd.read_csv(ROOT/f'{s}.csv',keep_default_na=False,dtype={'parent_recording_id':str,'final_group_id':str}) for s in ['train','val','test']}
mapping=json.loads((ROOT/'label_mapping.json').read_text())
label2id={str(k):int(v) for k,v in mapping['label2id'].items()}
id2label={int(k):str(v) for k,v in mapping['id2label'].items()}
assert len(label2id)==10 and set(label2id.values())==set(range(10))
assert all(id2label[i]==s for s,i in label2id.items())
assert 'Yellow-fronted_Barbet' in label2id and 'Ashy-headed_Laughingthrush' not in label2id
assert {s:len(df) for s,df in frames.items()}=={'train':29402,'val':925,'test':924}
required={'audio_path','species','label','split','parent_recording_id','final_group_id','augmentation_level','sha256','padding_fraction','quality_flags'}
all_paths=set()
for split,df in frames.items():
 assert required<=set(df.columns) and set(df.species)==set(label2id)
 assert (df.split==split).all() and df.audio_path.is_unique
 assert (df.label.astype(int)==df.species.map(label2id)).all()
 assert (df.final_group_id.str.len()>0).all() and (df.parent_recording_id.str.len()>0).all()
 for row in df.itertuples():
  path=(ROOT/row.audio_path).resolve()
  assert path.is_relative_to(ROOT.resolve()) and path.is_file()
  assert Path(row.audio_path).parts[:2] == (split,row.species), 'Path/split/species mismatch'
  assert row.audio_path not in all_paths;all_paths.add(row.audio_path)
  info=sf.info(path)
  assert (info.samplerate,info.channels,info.subtype,info.frames)==(16000,1,'PCM_16',80000)
  if VERIFY_AUDIO_HASHES:assert sha256(path)==row.sha256
for a,b in [('train','val'),('train','test'),('val','test')]:
 for key in ['parent_recording_id','final_group_id']:
  assert set(frames[a][key]).isdisjoint(set(frames[b][key])),(key,a,b)
train_df,val_df,test_metadata=frames['train'],frames['val'],frames['test']
save_json(RUN_DIR/'label_mapping.json',mapping)
save_json(RUN_DIR/'data_integrity.json',{'zip_sha256':ZIP_SHA256,'manifest_sha256':{s:sha256(ROOT/f'{s}.csv') for s in frames},'counts':{s:len(df) for s,df in frames.items()},'group_overlap':False})
print('Paths, labels, formats and group boundaries verified')


## 4. Matching model and feature extractor
Wav2Vec2 uses its waveform normalizer and a new classifier; freeze the convolutional frontend, fine-tune the transformer. Disable additional SpecAugment.
AST keeps the AudioSet checkpoint's 1024-frame/128-bin representation, positional embeddings and extractor normalization. The extractor pads five-second waveforms appropriately. No validation/test statistics are fitted.
Resolve the checkpoint revision once and save it. For exact future repeats, set MODEL_REVISION to the saved commit SHA.


In [ ]:
MODEL_REVISION=HfApi().model_info(CHECKPOINT).sha
extractor=AutoFeatureExtractor.from_pretrained(CHECKPOINT,revision=MODEL_REVISION)
assert extractor.sampling_rate==16000
ModelClass=Wav2Vec2ForSequenceClassification if MODEL_KIND=='wav2vec2' else ASTForAudioClassification
base_config=AutoConfig.from_pretrained(CHECKPOINT,revision=MODEL_REVISION)
assert base_config.model_type==('wav2vec2' if MODEL_KIND=='wav2vec2' else 'audio-spectrogram-transformer')
if MODEL_KIND=='ast':
 assert extractor.max_length==base_config.max_length==1024
 assert extractor.num_mel_bins==base_config.num_mel_bins==128
def new_model(dropout):
 config=AutoConfig.from_pretrained(CHECKPOINT,revision=MODEL_REVISION)
 config.num_labels=10;config.label2id=label2id;config.id2label=id2label
 config.problem_type='single_label_classification'
 if MODEL_KIND=='wav2vec2':
  config.hidden_dropout=dropout;config.attention_dropout=dropout
  config.activation_dropout=dropout;config.feat_proj_dropout=dropout
  config.final_dropout=dropout;config.layerdrop=0.;config.apply_spec_augment=False
 else:
  config.hidden_dropout_prob=dropout;config.attention_probs_dropout_prob=dropout
 model=ModelClass.from_pretrained(CHECKPOINT,revision=MODEL_REVISION,config=config,ignore_mismatched_sizes=True,attn_implementation='eager')
 if MODEL_KIND=='wav2vec2':model.freeze_feature_encoder()
 if GRADIENT_CHECKPOINTING and DEVICE.type=='cuda':
  if getattr(model,'supports_gradient_checkpointing',False): model.gradient_checkpointing_enable()
 return model.to(DEVICE)
versions={p:importlib.metadata.version(p) for p in ['torch','transformers','scikit-learn','optuna','numpy','soundfile','librosa']}
save_json(RUN_DIR/'environment.json',{'versions':versions,'python':platform.python_version(),'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None,'cpu_smoke_test':CPU_SMOKE_TEST,'amp_dtype':str(AMP_DTYPE),'seed':SEED,'checkpoint':CHECKPOINT,'revision':MODEL_REVISION,'full_cv':FULL_GROUPED_CV,'n_trials':N_TRIALS,'batch_choices':BATCH_CHOICES,'gradient_accumulation':ACCUMULATION_STEPS})
extractor.save_pretrained(RUN_DIR/'feature_extractor')


## 5. Loading and bounded class weights
For the active training subset only, use `w_c = clip(sqrt(N/(10*n_c)), 0.5, 2.0)`. This is square-root inverse frequency capped to prevent extreme weights. Counts include selected augmentations, not independent recordings. Recalculate weights for each fold's training rows; final fit uses the full training partition. Never calculate weights from validation/test. Do not additionally use a weighted sampler.


In [ ]:
def class_weights(df):
 counts=np.bincount(df.label.astype(int),minlength=10)
 assert np.all(counts>0),'Training subset missing a class'
 weights=np.clip(np.sqrt(len(df)/(10*counts)),.5,2.)
 return counts,torch.tensor(weights,dtype=torch.float32)
class AudioRows(Dataset):
 def __init__(self,df):self.df=df.reset_index(drop=True)
 def __len__(self):return len(self.df)
 def __getitem__(self,index):
  row=self.df.iloc[index];wave,sr=sf.read(ROOT/row.audio_path,dtype='float32')
  assert sr==16000 and wave.ndim==1 and len(wave)==80000
  return wave,int(row.label)
def collate(batch):
 waves,labels=zip(*batch);kwargs={'sampling_rate':16000,'return_tensors':'pt'}
 if MODEL_KIND=='wav2vec2':kwargs.update(padding=True,return_attention_mask=base_config.feat_extract_norm=='layer')
 return dict(extractor(list(waves),**kwargs)),torch.tensor(labels,dtype=torch.long)
def loader(df,batch_size,shuffle,seed=SEED):
 return DataLoader(AudioRows(df),batch_size=batch_size,shuffle=shuffle,collate_fn=collate,num_workers=NUM_WORKERS,generator=torch.Generator().manual_seed(seed),pin_memory=True)
def cleanup():
 gc.collect()
 if torch.cuda.is_available():torch.cuda.empty_cache()


## 6. Training-only group-stratified CV
Use deterministic StratifiedGroupKFold with shuffle=False; seed 42 controls model initialization, loaders and Optuna. Reserve entire groups, including every noisy sibling of a held-out clean clip. Score only clean rows in those held-out training groups, matching the clean-validation objective.
Practical mode scores fold 0 (a single grouped holdout, not a CV estimate); full CV reports mean and population standard deviation across three folds. Original val/test are not used in this search. These are selection scores, not unbiased final test estimates.


In [ ]:
sgkf=StratifiedGroupKFold(n_splits=3,shuffle=False)
folds=[];fold_export=[]
for fold,(tr_idx,hold_idx) in enumerate(sgkf.split(train_df,train_df.label,groups=train_df.final_group_id)):
 tr=train_df.iloc[tr_idx].copy();held=train_df.iloc[hold_idx].copy()
 va=held[held.augmentation_level=='clean'].copy()
 assert set(tr.final_group_id).isdisjoint(set(held.final_group_id))
 assert set(tr.parent_recording_id).isdisjoint(set(held.parent_recording_id))
 assert set(tr.label)==set(va.label)==set(range(10)),'Fold missing a species; inspect grouping'
 folds.append((tr,va))
 fold_export.extend({'audio_path':r.audio_path,'held_out_fold':fold,'final_group_id':r.final_group_id,'scored_as_clean_validation':r.augmentation_level=='clean'} for r in held.itertuples())
pd.DataFrame(fold_export).to_csv(RUN_DIR/'training_cv_folds.csv',index=False)
print([(len(tr),len(va)) for tr,va in folds])

# Select only clean examples from disjoint training-only group pools.
# Entire groups are assigned to a pool before sampling; unsampled siblings are unused.
active_train_df, active_val_df = train_df, val_df
if CPU_SMOKE_TEST:
 def balanced_smoke_subset(frame, per_class):
  clean=frame[frame.augmentation_level=='clean'].sort_values('audio_path')
  assert set(clean.label)==set(range(10)), 'Missing class in smoke-test pool'
  assert clean.groupby('label').size().min()>=per_class
  return clean.groupby('label',group_keys=False).sample(n=per_class,random_state=SEED).sort_values(['label','audio_path']).reset_index(drop=True)
 smoke_train=balanced_smoke_subset(folds[0][0],2)
 smoke_val=balanced_smoke_subset(folds[0][1],1)
 for key in ['parent_recording_id','final_group_id']:
  assert set(smoke_train[key]).isdisjoint(set(smoke_val[key]))
  for original in [val_df,test_metadata]:
   assert set(smoke_train[key]).isdisjoint(set(original[key]))
   assert set(smoke_val[key]).isdisjoint(set(original[key]))
 active_train_df,active_val_df=smoke_train,smoke_val
 folds=[(smoke_train,smoke_val)]
 smoke_train.to_csv(RUN_DIR/'smoke_train_selection.csv',index=False)
 smoke_val.to_csv(RUN_DIR/'smoke_validation_selection.csv',index=False)
 save_json(RUN_DIR/'smoke_experiment.json',{'preliminary':True,'comparable_to_full_results':False,
   'train_examples':len(smoke_train),'validation_examples':len(smoke_val),
   'validation_origin':'training-only grouped holdout','max_epochs':1,'batch_size':1,'trials':1})
 print('PRELIMINARY: 20 clean training examples; 10 training-only validation examples.')


## 7. AdamW, scheduling, early stopping and curves
Exclude biases and one-dimensional normalization parameters from weight decay. Cosine scheduler with 10% warmup. Gradient accumulation sums weighted losses, then divides gradients by total class-weight mass per update, including a final partial window. Unscale before clipping gradients to norm 1.0.
All metrics use ten labels. Validation loss uses weights computed from training. Best checkpoint and early stopping use validation macro F1. Checkpoints support inference/best-model loading, not exact mid-epoch optimizer resume.


In [ ]:
def metrics(y,pred):
 return {'macro_f1':float(f1_score(y,pred,labels=list(range(10)),average='macro',zero_division=0)),
         'accuracy':float(accuracy_score(y,pred))}
@torch.no_grad()
def evaluate(model,df,weights,batch_size):
 model.eval();ys=[];predictions=[];loss_sum=0.;weight_sum=0.
 device_weights=weights.to(DEVICE)
 loss_fn=nn.CrossEntropyLoss(weight=device_weights,reduction='sum')
 for features,labels in loader(df,batch_size,False):
  features={k:v.to(DEVICE,non_blocking=True) for k,v in features.items()};labels=labels.to(DEVICE)
  with amp_context():
   logits=model(**features).logits;loss=loss_fn(logits.float(),labels)
  loss_sum+=float(loss);weight_sum+=float(device_weights[labels].sum())
  ys.extend(labels.cpu().tolist());predictions.extend(logits.argmax(-1).cpu().tolist())
 return dict(metrics(ys,predictions),loss=loss_sum/weight_sum),ys,predictions

def plot_history(history,directory):
 df=pd.DataFrame(history);df.to_csv(directory/'history.csv',index=False)
 fig,axs=plt.subplots(1,2,figsize=(12,4))
 for column in ['train_loss','val_loss']:axs[0].plot(df.epoch,df[column],label=column)
 for column in ['train_macro_f1','val_macro_f1']:axs[1].plot(df.epoch,df[column],label=column)
 for ax in axs:ax.set_xlabel('Epoch');ax.legend();ax.grid(alpha=.2)
 axs[0].set_ylabel('Weighted cross entropy');axs[1].set_ylabel('Macro F1')
 fig.tight_layout();fig.savefig(directory/'learning_curves.png',dpi=160);plt.close(fig)

def fit(tr,va,hp,directory,seed):
 directory=Path(directory);directory.mkdir(parents=True,exist_ok=False)
 seed_everything(seed);model=None;optimizer=None
 counts,weights=class_weights(tr)
 save_json(directory/'class_weights.json',{'formula':'clip(sqrt(N/(10*n_c)),0.5,2.0)',
           'counts':counts.tolist(),'weights':weights.tolist(),'seed':seed})
 save_json(directory/'hyperparameters.json',hp)
 try:
  model=new_model(hp['dropout'])
  decay=[];no_decay=[]
  for name,param in model.named_parameters():
   if param.requires_grad:(no_decay if param.ndim<2 or name.endswith('.bias') else decay).append(param)
  optimizer=torch.optim.AdamW([{'params':decay,'weight_decay':hp['weight_decay']},
                              {'params':no_decay,'weight_decay':0.}],lr=hp['learning_rate'])
  train_loader=loader(tr,hp['batch_size'],True,seed)
  total_steps=math.ceil(len(train_loader)/ACCUMULATION_STEPS)*hp['max_epochs']
  scheduler=get_cosine_schedule_with_warmup(optimizer,int(.1*total_steps),total_steps)
  scaler=torch.amp.GradScaler(DEVICE.type,enabled=AMP_ENABLED and AMP_DTYPE==torch.float16)
  device_weights=weights.to(DEVICE)
  loss_fn=nn.CrossEntropyLoss(weight=device_weights,reduction='sum')
  history=[];best=-1.;stale=0;best_epoch=0
  for epoch in range(1,hp['max_epochs']+1):
   model.train();optimizer.zero_grad(set_to_none=True)
   total_loss=0.;total_mass=0.;window_mass=0.;ys=[];ps=[]
   for step,(features,labels) in enumerate(train_loader,1):
    features={k:v.to(DEVICE,non_blocking=True) for k,v in features.items()};labels=labels.to(DEVICE)
    with amp_context():
     logits=model(**features).logits;loss=loss_fn(logits.float(),labels)
    if not torch.isfinite(loss):raise RuntimeError('Nonfinite loss; stop and inspect data/settings')
    scaler.scale(loss).backward()
    mass=float(device_weights[labels].sum());window_mass+=mass
    total_mass+=mass;total_loss+=float(loss.detach())
    ys.extend(labels.cpu().tolist());ps.extend(logits.detach().argmax(-1).cpu().tolist())
    if step%ACCUMULATION_STEPS==0 or step==len(train_loader):
     scaler.unscale_(optimizer)
     for parameter in model.parameters():
      if parameter.grad is not None:parameter.grad.div_(window_mass)
     torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
     old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()
     if scaler.get_scale()>=old_scale:scheduler.step()
     optimizer.zero_grad(set_to_none=True);window_mass=0.
   vm,_,_=evaluate(model,va,weights,hp['batch_size'])
   history.append({'epoch':epoch,'train_loss':total_loss/total_mass,'val_loss':vm['loss'],
                   'train_macro_f1':metrics(ys,ps)['macro_f1'],'val_macro_f1':vm['macro_f1'],'val_accuracy':vm['accuracy']})
   plot_history(history,directory);print(directory.name,history[-1])
   if vm['macro_f1']>best+MIN_F1_IMPROVEMENT:
    best=vm['macro_f1'];best_epoch=epoch;stale=0
    model.save_pretrained(directory/'best_model',safe_serialization=True)
    extractor.save_pretrained(directory/'best_model')
    save_json(directory/'best_metrics.json',dict(vm,best_epoch=best_epoch))
   else:stale+=1
   if stale>=EARLY_STOPPING_PATIENCE:break
  model.load_state_dict(load_file(str(directory/'best_model/model.safetensors'),device='cpu'))
  return model,weights,{'macro_f1':best,'best_epoch':best_epoch,'epochs_run':len(history)}
 except Exception:
  if model is not None:model.cpu()
  raise
 finally:
  del optimizer
  cleanup()


## 8. Optuna - explicit tuning execution
Enable RUN_TUNING in settings. Sequential TPE with seed 42 searches all required categorical values. GPU OOM trials are pruned and logged; if none complete, begin a new run with BATCH_CHOICES=[4]. Do not silently alter a trial's batch size. Effective batch is up to batch_size - ACCUMULATION_STEPS.
Full CV can require three fits per trial; practical mode requires one. Runtime/hardware differences can still affect reproducibility. Trial scores and fold standard deviations are saved to Drive; none are invented.


In [ ]:
SEARCH_SPACE={'learning_rate':[1e-5,2e-5,3e-5,5e-5],'dropout':[.1,.2,.3],
 'max_epochs':[5,10,15],'batch_size':BATCH_CHOICES,'weight_decay':[.01,.05,.1]}
if CPU_SMOKE_TEST:
 SEARCH_SPACE={'learning_rate':[2e-5],'dropout':[.1],'max_epochs':[1],'batch_size':[1],'weight_decay':[.01]}
save_json(RUN_DIR/'search_space.json',SEARCH_SPACE)
def objective(trial):
 hp={name:trial.suggest_categorical(name,choices) for name,choices in SEARCH_SPACE.items()}
 scores=[];model=None
 try:
  for fold in (range(3) if FULL_GROUPED_CV else [0]):
   tr,va=folds[fold]
   model,weights,result=fit(tr,va,hp,RUN_DIR/'trials'/f'trial_{trial.number:03d}_fold_{fold}',SEED+fold)
   scores.append(result['macro_f1']);model.cpu();del model;model=None;cleanup()
  trial.set_user_attr('fold_macro_f1',scores)
  trial.set_user_attr('mean_macro_f1',float(np.mean(scores)))
  trial.set_user_attr('std_macro_f1',float(np.std(scores)) if FULL_GROUPED_CV else None)
  return float(np.mean(scores))
 except RuntimeError as exc:
  if 'out of memory' in str(exc).lower():
   trial.set_user_attr('failure_reason','GPU OOM: batch configuration infeasible on this GPU')
   raise optuna.TrialPruned('GPU OOM; try batch choices [4] in a fresh run') from None
  raise
 finally:
  if model is not None:model.cpu();del model
  cleanup()
if RUN_TUNING:
 study=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=SEED))
 try:study.optimize(objective,n_trials=N_TRIALS,n_jobs=1,gc_after_trial=True)
 finally:study.trials_dataframe().to_csv(RUN_DIR/'optuna_trials.csv',index=False)
 complete=[t for t in study.trials if t.state==optuna.trial.TrialState.COMPLETE]
 if not complete:raise RuntimeError('No trial completed. Inspect OOM logs; start fresh with BATCH_CHOICES=[4].')
 BEST_HP=dict(study.best_trial.params)
 save_json(RUN_DIR/'selected_hyperparameters.json',{'hyperparameters':BEST_HP,
  'objective':'training-only grouped CV macro F1' if FULL_GROUPED_CV else 'training-only grouped holdout macro F1',
  'best_trial':study.best_trial.number,'score':study.best_value,
  'fold_scores':study.best_trial.user_attrs['fold_macro_f1'],'fold_std':study.best_trial.user_attrs['std_macro_f1']})
 display(study.trials_dataframe())
else:print('Tuning not run. Enable RUN_TUNING, then rerun this cell.')


## 9. Final training and original validation evaluation
Enable RUN_FINAL_TRAINING after inspecting tuning results. Hyperparameters are selected using only training-partition folds. The original validation set selects the final early-stopping checkpoint; the test set remains unevaluated. Artifacts use separate model-specific, timestamped Drive directories. Training-curve training metrics are collected during updates, whereas validation metrics use the epoch-end model.

In CPU smoke mode this optional cell repeats the selected one-epoch experiment on the same small training-only holdout, NOT the full training or original validation set. Tuning alone already saves a successful smoke checkpoint and loss/accuracy/macro-F1 history.


In [ ]:
def save_evaluation(model, frame, weights, hp, directory, name):
    result, truth, predicted = evaluate(model, frame, weights, hp['batch_size'])
    names = [id2label[i] for i in range(10)]
    report = classification_report(truth, predicted, labels=list(range(10)),
                                   target_names=names, output_dict=True, zero_division=0)
    save_json(directory/f'{name}_metrics.json', result)
    save_json(directory/f'{name}_classification_report.json', report)
    per_species = pd.DataFrame(report).T.loc[names]
    per_species.to_csv(directory/f'{name}_per_species.csv', index_label='species')
    predictions = frame[['audio_path','species','label','parent_recording_id','final_group_id']].copy()
    predictions['predicted_label'] = predicted
    predictions.to_csv(directory/f'{name}_predictions.csv', index=False)
    matrix = confusion_matrix(truth, predicted, labels=list(range(10)))
    pd.DataFrame(matrix,index=names,columns=names).to_csv(directory/f'{name}_confusion_matrix.csv')
    fig, ax = plt.subplots(figsize=(12,11))
    ConfusionMatrixDisplay(matrix,display_labels=names).plot(ax=ax,xticks_rotation=90,colorbar=False)
    fig.tight_layout();fig.savefig(directory/f'{name}_confusion_matrix.png',dpi=150);plt.show();plt.close(fig)
    fig, ax = plt.subplots(figsize=(12,5))
    per_species[['precision','recall']].plot.bar(ax=ax,ylim=(0,1))
    fig.tight_layout();fig.savefig(directory/f'{name}_precision_recall.png',dpi=150);plt.show();plt.close(fig)
    display(per_species);print(result)
    return result

FINAL_DIR = RUN_DIR/'final'
if RUN_FINAL_TRAINING:
    assert (RUN_DIR/'selected_hyperparameters.json').is_file(), 'Complete tuning first.'
    BEST_HP = json.loads((RUN_DIR/'selected_hyperparameters.json').read_text())['hyperparameters']
    final_model = None
    try:
        final_model, final_weights, final_result = fit(active_train_df,active_val_df,BEST_HP,FINAL_DIR,SEED)
        validation_metrics = save_evaluation(final_model,active_val_df,final_weights,BEST_HP,FINAL_DIR,'smoke_validation' if CPU_SMOKE_TEST else 'validation')
        save_json(FINAL_DIR/'selection_finalized.json', {
            'hyperparameters':BEST_HP,'seed':SEED,'best_epoch':final_result['best_epoch'],
            'checkpoint_revision':MODEL_REVISION,'package_sha256':sha256(ZIP_ON_DRIVE),
            'test_used_for_selection':False,'preliminary_cpu_smoke_test':CPU_SMOKE_TEST})
    except RuntimeError as exc:
        if 'out of memory' in str(exc).lower():
            raise RuntimeError('Final training ran out of GPU memory. Do not run test. Start a new run with feasible batch choices.') from exc
        raise
    finally:
        if final_model is not None: final_model.cpu();del final_model
        cleanup()
else:
    print('Final training disabled. Enable RUN_FINAL_TRAINING after tuning.')


## 10. Explicit final test evaluation - run once after model selection
Do not enable this cell until the configuration and validation-selected checkpoint are final. Test results must not guide subsequent hyperparameter choices. This cell refuses to overwrite an existing test evaluation. It loads the frozen checkpoint rather than retraining. Validation and test stay clean; this does not measure independent-noise robustness.


In [ ]:
RUN_FINAL_TEST = False  # Explicitly change only after model selection is finalized.
if RUN_FINAL_TEST:
    assert not CPU_SMOKE_TEST, 'Smoke tests never evaluate the original test set.'
    assert (FINAL_DIR/'selection_finalized.json').is_file(), 'Finalize training/validation first.'
    assert not (FINAL_DIR/'test_metrics.json').exists(), 'Test already evaluated; inspect saved results.'
    frozen = json.loads((FINAL_DIR/'selection_finalized.json').read_text())
    hp = frozen['hyperparameters']
    weights = torch.tensor(json.loads((FINAL_DIR/'class_weights.json').read_text())['weights'],dtype=torch.float32)
    test_model = ModelClass.from_pretrained(FINAL_DIR/'best_model',attn_implementation='eager').to(DEVICE)
    try:
        test_metrics = save_evaluation(test_model,test_metadata,weights,hp,FINAL_DIR,'test')
    finally:
        test_model.cpu();del test_model;cleanup()
else:
    print('Test remains locked. No test predictions or metrics computed.')


## 11. Upload inference
After training, optionally upload an audio file. Decode to mono 16 kHz in memory, split into non-overlapping five-second windows, and zero-pad the final short window. No pitch/speed changes or noise augmentation. Report segment predictions and a source-duration-weighted mean of softmax scores. Scores are uncalibrated model confidence, not proof of species presence; this closed-set model has no unknown-species rejection. Silent uploads are not classified. Low-energy or background-only audio may still produce misleading confidence.


In [ ]:
RUN_UPLOAD_INFERENCE = False
if RUN_UPLOAD_INFERENCE:
    import librosa
    assert (FINAL_DIR/'selection_finalized.json').is_file(), 'Train a model first.'
    inference_extractor = AutoFeatureExtractor.from_pretrained(FINAL_DIR/'best_model')
    inference_model = ModelClass.from_pretrained(FINAL_DIR/'best_model',attn_implementation='eager').to(DEVICE).eval()
    try:
        uploaded = files.upload()
        for filename, payload in uploaded.items():
            audio, sr = librosa.load(io.BytesIO(payload),sr=16000,mono=True)
            if not len(audio) or not np.isfinite(audio).all() or not np.any(audio):
                print(filename, 'empty, invalid or digitally silent; no prediction');continue
            scores=[];durations=[];segments=[]
            for start in range(0,len(audio),80000):
                segment = audio[start:start+80000]; actual = len(segment)
                if not np.any(segment): continue
                segment = np.pad(segment,(0,80000-actual))
                kwargs = {'padding':True,'return_attention_mask':base_config.feat_extract_norm=='layer'} if MODEL_KIND=='wav2vec2' else {}
                inputs = inference_extractor([segment],sampling_rate=16000,return_tensors='pt',**kwargs)
                inputs = {key:value.to(DEVICE) for key,value in inputs.items()}
                with torch.inference_mode(), amp_context():
                    probabilities = inference_model(**inputs).logits.float().softmax(-1)[0].cpu().numpy()
                scores.append(probabilities);durations.append(actual)
                label = int(probabilities.argmax())
                segments.append({'start_seconds':start/16000,'content_seconds':actual/16000,
                                 'species':id2label[label],'confidence':float(probabilities[label])})
            combined = np.average(scores,axis=0,weights=durations)
            print(filename);display(pd.DataFrame(segments))
            display(pd.DataFrame({'species':[id2label[i] for i in range(10)],'confidence':combined}).sort_values('confidence',ascending=False))
    finally:
        inference_model.cpu();del inference_model;cleanup()


## Interpretation and references
These notebooks were generated and syntax-checked without model training. No accuracy or F1 results are claimed. Quality/provenance flags remain in the manifests; energy is not a bird-presence annotation. Recording-group separation respects known and precautionary groups, but unknown original-event provenance remains unresolved. Six single-source noise categories do not establish independent-noise generalization. Original clean validation/test sets are used as supplied.

API references: [Wav2Vec2](https://huggingface.co/docs/transformers/v4.57.1/model_doc/wav2vec2), [AST](https://huggingface.co/docs/transformers/v4.57.1/model_doc/audio-spectrogram-transformer), [StratifiedGroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html), [Optuna reproducibility](https://optuna.readthedocs.io/en/stable/faq.html).
